In [ ]:
pip install gymnasium numpy

In [ ]:
%pip install pygame

In [ ]:
import gymnasium as gym
import numpy as np
import random

# 1. Inisialisasi Environment
# is_slippery=False membuat pergerakan agen jadi deterministik (lebih mudah dipelajari)
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode=None)
# human untuk visualisai pembelajaran
# env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human")

# 2. Inisialisasi Q-Table dengan nol
state_size = env.observation_space.n
action_size = env.action_space.n
q_table = np.zeros((state_size, action_size))

# 3. Hyperparameters 
learning_rate = 0.8    
discount_factor = 0.95 
epsilon = 1.0          
epsilon_decay = 0.001  
total_episodes = 1000

# 4. Algoritma Q-Learning
for episode in range(total_episodes):
    state, info = env.reset()
    done = False

    for step in range(100):
        # Action selection (Epsilon-greedy) 
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample() # Eksplorasi: pilih acak
        else:
            action = np.argmax(q_table[state, :]) # Eksploitasi: pilih yang terbaik di Q-Table

        # Melakukan aksi
        new_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # Update Q-Table menggunakan Persamaan Bellman
        q_table[state, action] = q_table[state, action] + learning_rate * (
            reward + discount_factor * np.max(q_table[new_state, :]) - q_table[state, action]
        )
        
        state = new_state
        if done:
            break
            
    # Kurangi epsilon agar agen lebih fokus pada apa yang sudah dipelajari

print("Pelatihan selesai!")
print("Q-Table Hasil Belajar:")
print(q_table)
env.close()

In [ ]:
# Tes hasil latihan

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from IPython import display
import time

# 1. Inisialisasi Environment khusus untuk Visualisasi di Notebook
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")

# 2. Reset Environment untuk memulai game baru
state, info = env.reset()
done = False
total_rewards = 0

# Menyiapkan area plot di Jupyter Notebook
plt.figure(figsize=(5, 5))
tampilan_layar = plt.imshow(env.render())
plt.axis('off')

print("Agen mulai berjalan menuju Goal...")

# 3. Loop Permainan
for step in range(100):
    # Agen SELALU memilih aksi terbaik dari Q-Table yang sudah dilatih (Eksploitasi penuh)
    action = np.argmax(q_table[state, :])
    
    # Lakukan aksi
    new_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_rewards += reward
    state = new_state
    
    # --- PROSES ANIMASI DI NOTEBOOK ---
    # Update gambar frame terbaru
    tampilan_layar.set_data(env.render())
    
    # Gambar ulang di sel Jupyter yang sama (membuat efek animasi)
    display.display(plt.gcf())
    display.clear_output(wait=True)
    
    # Beri jeda sedikit agar bisa mengikuti pergerakan agen (0.2 detik)
    time.sleep(0.2) 
    
    if done:
        if reward == 1:
            print(f"Sukses! Agen berhasil mencapai GOAL dalam {step+1} langkah.")
        else:
            print("Agen masuk ke dalam LUBANG.")
        break

env.close()